## TAREA 5 - Fisica Computacional II

Nombre completo: ______________________  RUT: ______________

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle

In [ ]:
class CatalogoEstelar:
    # Clase para leer, limpiar y analizar un catalogo estelar

    def __init__(self, ruta_csv):
        # Lectura del csv y columnas como arreglos independientes de numpy
        df = pd.read_csv(ruta_csv)
        self.ids = df.iloc[:, 0].to_numpy(dtype=str)
        self.temperatura = pd.to_numeric(df['temperatura_K'], errors='coerce').to_numpy(dtype=float)
        self.luminosidad = pd.to_numeric(df['luminosidad_Lsol'], errors='coerce').to_numpy(dtype=float)
        self.distancia = pd.to_numeric(df['distancia_pc'], errors='coerce').to_numpy(dtype=float)
        # Limpieza automatica
        self.limpiar_datos()

    def limpiar_datos(self):
        # Valores finitos y positivos en las tres propiedades fisicas
        validos = np.ones(len(self.ids), dtype=bool)
        for col in (self.temperatura, self.luminosidad, self.distancia):
            validos &= np.isfinite(col) & (col > 0)

        # Duplicados: se conserva solo la primera aparicion de cada id
        _, primeros = np.unique(self.ids, return_index=True)
        no_dup = np.zeros(len(self.ids), dtype=bool)
        no_dup[primeros] = True

        # Se aplica la misma mascara a todas las columnas
        mascara = validos & no_dup
        self.ids = self.ids[mascara]
        self.temperatura = self.temperatura[mascara]
        self.luminosidad = self.luminosidad[mascara]
        self.distancia = self.distancia[mascara]

    def resumen(self):
        # Calculos vectorizados sobre el catalogo limpio
        print('=== Resumen del catalogo limpio ===')
        print(f'Numero de estrellas: {len(self.ids)}')
        print(f'Temperatura media: {np.mean(self.temperatura):.2f} K')
        print(f'Temperatura maxima: {np.max(self.temperatura):.2f} K')
        print(f'Luminosidad mediana: {np.median(self.luminosidad):.2f} Lsol')
        print(f'Distancia minima: {np.min(self.distancia):.2f} pc')
        print(f'Distancia maxima: {np.max(self.distancia):.2f} pc')
        print(f'Estrella mas caliente: {self.ids[np.argmax(self.temperatura)]}')
        print(f'Estrella mas cercana: {self.ids[np.argmin(self.distancia)]}')

    def _directorio(self):
        # Crea resultados/ si no existe (lo usa cada metodo de guardado)
        carpeta = Path('resultados')
        carpeta.mkdir(exist_ok=True)
        return carpeta

    # ---------- CSV ----------
    def guardar_csv(self):
        ruta = self._directorio() / 'catalogo_copia.csv'
        df = pd.DataFrame({'id': self.ids,
                           'temperatura_K': self.temperatura,
                           'luminosidad_Lsol': self.luminosidad,
                           'distancia_pc': self.distancia})
        df.to_csv(ruta, index=False)

    def cargar_csv(self):
        # Se lee con el constructor de la clase
        return CatalogoEstelar('resultados/catalogo_copia.csv')

    # ---------- NPY ----------
    def guardar_npy(self):
        np.save(self._directorio() / 'temperaturas.npy', self.temperatura)

    def cargar_npy(self):
        return np.load('resultados/temperaturas.npy')

    # ---------- NPZ ----------
    def guardar_npz(self):
        np.savez_compressed(self._directorio() / 'catalogo_completo.npz',
                            ids=self.ids,
                            temperatura=self.temperatura,
                            luminosidad=self.luminosidad,
                            distancia=self.distancia)

    def cargar_npz(self):
        # Sin allow_pickle porque los ids son arreglo de texto de numpy
        return np.load('resultados/catalogo_completo.npz')

    # ---------- Pickle ----------
    def guardar_pickle(self):
        datos = {'ids': self.ids,
                 'temperatura': self.temperatura,
                 'luminosidad': self.luminosidad,
                 'distancia': self.distancia}
        with open(self._directorio() / 'catalogo_completo.pkl', 'wb') as f:
            pickle.dump(datos, f)

    def cargar_pickle(self):
        # Solo abrir pickle creados por uno mismo o de fuentes confiables
        with open('resultados/catalogo_completo.pkl', 'rb') as f:
            return pickle.load(f)

    # ---------- Verificacion ----------
    def verificar_archivos(self):
        n = len(self.ids)

        # CSV
        c = self.cargar_csv()
        ok = (len(c.ids) == n and np.array_equal(self.ids, c.ids)
              and np.allclose(self.temperatura, c.temperatura)
              and np.allclose(self.luminosidad, c.luminosidad)
              and np.allclose(self.distancia, c.distancia))
        print(f'CSV coincide: {ok}')

        # NPY (solo temperaturas)
        t = self.cargar_npy()
        ok = len(t) == n and np.allclose(self.temperatura, t)
        print(f'NPY coincide: {ok}')

        # NPZ
        z = self.cargar_npz()
        ok = (len(z['ids']) == n and np.array_equal(self.ids, z['ids'])
              and np.allclose(self.temperatura, z['temperatura'])
              and np.allclose(self.luminosidad, z['luminosidad'])
              and np.allclose(self.distancia, z['distancia']))
        print(f'NPZ coincide: {ok}')

        # Pickle
        p = self.cargar_pickle()
        ok = (len(p['ids']) == n and np.array_equal(self.ids, p['ids'])
              and np.allclose(self.temperatura, p['temperatura'])
              and np.allclose(self.luminosidad, p['luminosidad'])
              and np.allclose(self.distancia, p['distancia']))
        print(f'Pickle coincide: {ok}')

    def comparar_formatos(self):
        # Tamano en bytes de cada archivo en una tabla
        archivos = ['catalogo_copia.csv', 'temperaturas.npy',
                    'catalogo_completo.npz', 'catalogo_completo.pkl']
        tabla = pd.DataFrame({'Archivo': archivos,
                              'Tamano (bytes)': [Path('resultados', a).stat().st_size for a in archivos]})
        print(tabla.to_string(index=False))

In [ ]:
# Uso de la clase
cat = CatalogoEstelar('catalogo_estelar.csv')
cat.resumen()

# Guardado en todos los formatos
cat.guardar_csv()
cat.guardar_npy()
cat.guardar_npz()
cat.guardar_pickle()

# Verificacion y comparacion
cat.verificar_archivos()
cat.comparar_formatos()

## Preguntas teoricas

**Diferencias entre CSV, NPY, NPZ y Pickle**

- CSV: texto plano legible y compatible con casi cualquier programa, pero es el que mas pesa y es mas lento de leer.
- NPY: binario de numpy que guarda un solo arreglo. Es el mas pequeno y rapido, pero solo guarda una columna.
- NPZ: contenedor comprimido de varios arreglos de numpy. Pesa poco y guarda todo el catalogo.
- Pickle: guarda cualquier objeto de Python (por ejemplo un diccionario). Es propio de Python y no se debe abrir si no es de una fuente confiable.

**Formato para compartir el catalogo con otros usuarios**

CSV, porque es universal: se abre en Excel, en otros lenguajes y se puede leer sin Python, aunque pese mas.

**Formato para varios arreglos NumPy en un solo archivo**

NPZ, porque esta hecho para eso: guarda varios arreglos con nombre, comprimidos, y no necesita pickle ni allow_pickle.